In [ ]:
import csv
import pandas as pd
import random
import itertools
import numpy as np
from google.colab import drive
import re

## Guide
### Instructions
1) Dormspam the Tuesday a week prior to the week of the event.
2) Close reservation form Sunday at midnight.
3) Add all kerbs in current reservation form to the "tracking.csv" file. Be sure to indicate the current event.
4) Add all kerbs that were accepted in the last event to the "accepted.csv" file. Be sure to indicate what the last event was.
5) Import a csv file of the reservations for the current event. Rename the relevant column names to "seats", "names", "kerbs", and "allergies".
6) Rename the file and update the variable in the code block below. For example, if I renamed the file to "mince_dec.csv", I would update the file variable to "mince_dec.csv".
7) Run all code blocks.
8) Manually check the selected guests to ensure that they don't have any relevant dietary restrictions.

### Rules
* The weight of a group of people is equivalent to the average of all individual weights in the group
* Once someone receives a spot in a Pop up, their weight resets to 0

### Key
1) **bang**: Flung out of Space
2) **armenia**: Armenia
3) **fall**: Fall in New England
4) **art**: Night at the Art Gallery


### Update these variables!

In [ ]:
events = [
    "bang", "armenia", "fall", "art",
    "india", "datamatch", "picnic",
    "tea", "alice", "earth",
    "lny", "japan", "italy",
    "garden", "supper", "brazil",
    "ghibli", "swotw", "africa",
    "untitled", "ayuy", "entropy"
]
events = {e: i for i, e in enumerate(events)}

In [ ]:
drive.mount('/content/drive', force_remount=True) #or however works best for u

Mounted at /content/drive


### Create log.csv
The log.csv file records the number of times someone has registered for a MINCE Pop up, how many MINCE Pop ups they've attended, the last event they attended, and how many lottery tickets they currently hold.

In [ ]:
class Guest:
    def __init__(self, kerb):
        self.kerb = kerb
        self.count = 0
        self.accepted = 0
        self.current_weight = 0
        self.last_attended = None



df_tracking = pd.read_csv('/content/drive/My Drive/[MINCE] General/master documents/lottery/tracking.csv')
df_accepted = pd.read_csv('/content/drive/My Drive/[MINCE] General/master documents/lottery/accepted.csv')

log_tracking = {}

def parse_kerbs_str(kerbs_str):
  kerbs = re.split('[;,]', kerbs_str)

  out = []
  for kerb in kerbs:
      kerb = kerb.lower().strip()
      if not kerb:
        continue
      if not (kerb.endswith('@mit.edu') or '@' in kerb) :
          kerb += '@mit.edu'
      out.append(kerb)
  return out

def extract_kerbs(df):
    for k, event in zip(df['kerb'], df['event']):
      if type(k) != str:
          continue
      kerbs = parse_kerbs_str(k)
      for kerb in kerbs:
          if kerb not in log_tracking:
            log_tracking[kerb] = Guest(kerb)
          yield kerb, event


# first we look at who was accepted before
for kerb, event in extract_kerbs(df_accepted):
    log_tracking[kerb].accepted += 1
    log_tracking[kerb].last_attended = event
    print(kerb)

for kerb, event in extract_kerbs(df_tracking):
    log_tracking[kerb].count += 1
    log_tracking[kerb].current_weight += 1
    if log_tracking[kerb].last_attended == event: #Reset for last attended
        log_tracking[kerb].current_weight = 0
    print(kerb)

guests = []
for kerb, guest in log_tracking.items():
  weight = guest.current_weight
  guests.append([kerb, guest.count, guest.accepted, weight, guest.last_attended])

df_log = pd.DataFrame(guests, columns=["kerbs", "counts", "accepted", "weight", "last attended"])
df_log.to_csv('log.csv')

iliu@mit.edu
jszhu@mit.edu
sabchen@mit.edu
louisea@mit.edu
eyosias@mit.edu
tafeyan@gmail.com
beverlym@mit.edu
janicey@mit.edu
maxfan@mit.edu
kjulca@mit.edu
dogbe@mit.edu
azf@mit.edu
moralejo@mit.edu
laiwachu@mit.edu
handales@mit.edu
timberc@mit.edu
evag@mit.edu
katwli@mit.edu
katwli@mit.edu
yazana@mit.edu
mgeo@mit.edu
wendyz@mit.edu
iliao@mit.edu
abelyan@mit.edu
grigt@mit.edu
mshafim@mit.edu
jcahaly@mit.edu
mtang17@mit.edu
ptimil@mit.edu
zabroves@mit.edu
mnatalie@mit.edu
armen@sloan.mit.edu
ramirjos@mit.edu
diao@mit.edu
emmachen@mit.edu
jacobrod@mit.edu
nyouyang@mit.edu
mzetina@mit.edu
miglesia@mit.edu
yanswu@mit.edu
spruce@mit.edu
greerj@mit.edu
teejay@mit.edu
ayoub@mit.edu
jocelin@mit.edu
curtiss@mit.edu
jmfortt@mit.edu
alisonw@mit.edu
smthom@mit.edu
rya@mit.edu
emilywg@mit.edu
chrswang@mit.edu
mqwang@mit.edu
michli@mit.edu
pronina@mit.edu
lcui@mit.edu
solcu483@mit.edu
fmn@mit.edu
gminnout@mit.edu
cindyxie@mit.edu
claraxu@mit.edu
wangcx@mit.edu
chxchen@mit.edu
hgazdus@mit.edu
chriswu

In [ ]:
log_tracking['caseyf@mit.edu'].current_weight

8

In [ ]:
#df = pd.read_csv('mince_dec.csv')
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('entropy.csv')

high_weight_cutoff = 7

def parse_kerbs_str(kerbs_str):
  if pd.isna(kerbs_str):
      return []
  kerbs_str = str(kerbs_str)

  kerbs = re.split('[;,]', kerbs_str)

  out = []
  for kerb in kerbs:
      kerb = kerb.lower().strip()
      if not kerb:
        continue
      if not (kerb.endswith('@mit.edu') or '@' in kerb) :
          kerb += '@mit.edu'
      out.append(kerb)
  return out

def process_kerbs(kerbs_str):
    kerbs = parse_kerbs_str(kerbs_str) # Calls the parse_kerbs_str defined above

    weights = []
    is_pref = True
    for kerb in kerbs:
        if kerb not in log_tracking:
          weights.append(1)
          continue
        weights.append(log_tracking[kerb].current_weight)
        if log_tracking[kerb].last_attended is not None:
          is_pref = False

    # Avoid ZeroDivisionError if weights list is empty
    avg_weight = sum(weights) / len(weights) if weights else 0

    # is_pref = any(w >= high_weight_cutoff for w in weights)

    return ','.join(kerbs), ','.join(map(str, weights)), (avg_weight, int(is_pref))

def update_row(row):
    # Create a new dictionary from the row to ensure modifications don't interfere with original df
    new_data = row.to_dict()

    kerbs_str_processed, weight_str_processed, rest = process_kerbs(row['kerbs'])

    avg_weight, pref = rest

    new_data['kerbs'] = kerbs_str_processed
    new_data['weights'] = weight_str_processed
    new_data['avg_weight'] = avg_weight
    new_data['pref'] = pref

    # Return a new Series for df.apply to properly construct the updated DataFrame
    return pd.Series(new_data)

df = df.apply(update_row, axis=1)
df

Saving entropy.csv to entropy (2).csv


,seats,names,kerbs,allergies,weights,avg_weight,pref
0,2,Calista Huang,calistah@mit.edu,No,4,4.0,1
1,1,Nicole Zheng,nmzheng@mit.edu,No,2,2.0,1
2,1,Isabella Wu,isawu888@mit.edu,No,1,1.0,1
3,1,Sydney Saenz,saenz728@mit.edu,No,3,3.0,1
4,1,Akpandu Ekezie,akpandu@mit.edu,No,1,1.0,1
...,...,...,...,...,...,...,...
767,2,"Lindsay Reyes, Jessie Lin","linmari4@mit.edu,jessjlin@mit.edu",No,"5,1",3.0,0
768,2,"Bill Chen, Anna Gluschuk","runxi@mit.edu,annag26@mit.edu",No,"1,1",1.0,1
769,1,Emily Wong,emilywg@mit.edu,No,1,1.0,0
770,2,"Tiki Wu, Joyce Liu","twu@mit.edu,jliu07@mit.edu",No,"2,2",2.0,1


In [ ]:

pref_df, normal_df = df[df['pref'] == 1], df[df['pref'] != 1]

# Handle cases where avg_weight might be zero for sampling
# Add a small epsilon to zero weights to make them positive for the sample function
pref_df['sampling_weight'] = pref_df['avg_weight'].apply(lambda x: x if x > 0 else 0.001)
normal_df['sampling_weight'] = normal_df['avg_weight'].apply(lambda x: x if x > 0 else 0.001)

pref_df = pref_df.sample(frac=1, weights = pref_df['sampling_weight'], random_state=42) # Weight and shuffle prefs
lottery_df = normal_df.sample(frac=1, weights=normal_df['sampling_weight'], random_state=42) # Weight and shuffle normal
final_df = pd.concat([pref_df, lottery_df]).reset_index(drop=True)
final_df

/tmp/ipykernel_534/3470849919.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pref_df['sampling_weight'] = pref_df['avg_weight'].apply(lambda x: x if x > 0 else 0.001)
/tmp/ipykernel_534/3470849919.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  normal_df['sampling_weight'] = normal_df['avg_weight'].apply(lambda x: x if x > 0 else 0.001)


,seats,names,kerbs,allergies,weights,avg_weight,pref,sampling_weight
0,1,Peter Berggren,pmberg@mit.edu,No,4,4.0,1,4.000
1,2,"Megan Tian, Kayla Truong","megant@mit.edu,ktruong4@mit.edu",No,"7,1",4.0,1,4.000
2,2,"Melody Yu, Jason Youm","yumelody@mit.edu,jyoum@mit.edu",No,"2,2",2.0,1,2.000
3,2,"Louisa Zhu, Francis Liu","looeysa@mit.edu,fliu5@mit.edu",No,"3,1",2.0,1,2.000
4,1,Vernon Lin,vsl@mit.edu,No,3,3.0,1,3.000
...,...,...,...,...,...,...,...,...
767,2,"Hager Gomaa, Vaibhavi Addala","hagerg57@mit.edu,vaddala7@mit.edu",No,"0,0",0.0,0,0.001
768,1,Arielsie Li,arielsie@mit.edu,No,0,0.0,0,0.001
769,2,"Angela Wu, Yaqi Li","aywu@mit.edu,yaqili@mit.edu",No,"0,0",0.0,0,0.001
770,2,"Michelle Mo, Ellie Lei","michmo@mit.edu,ellei89@mit.edu",No,"0,0",0.0,0,0.001


In [ ]:
expanded_rows = []

for _, row in final_df.iterrows():
    names = [n.strip() for n in str(row['names']).split(',')]
    kerbs = [k.strip() for k in str(row['kerbs']).split(',')]
    weights = [w.strip() for w in str(row['weights']).split(',')] if pd.notnull(row['weights']) else ["" for _ in kerbs]

    for i in range(len(names)):
        entry = {
            'seats': row['seats'] if i == 0 else '',
            'name': names[i] if i < len(names) else '',
            'kerb': kerbs[i] if i < len(kerbs) else '',
            'weight': weights[i] if i < len(weights) else '',
            'avg_weight': row['avg_weight'],
            'allergies': row['allergies'],
            'pref': row['pref'],
            'last_attended': log_tracking[kerbs[i]].last_attended if (i < len(kerbs) and kerbs[i] in log_tracking) else None
        }
        expanded_rows.append(entry)

expanded_df = pd.DataFrame(expanded_rows)
expanded_df.to_csv('winners.csv')
expanded_df

,seats,name,kerb,weight,avg_weight,allergies,pref,last_attended
0,1,Joy Ren,joyren@mit.edu,2,2.0,No,1,None
1,1,Ansia Sae,ansia@mit.edu,1,1.0,No,1,None
2,1,Alyssa Garger,alyssabg@mit.edu,1,1.0,allergic to hazelnuts,1,None
3,2,Angeline Peng,pengs14@mit.edu,8,5.0,No,1,None
4,,Mohamed Wacyl Meddour,meddour@mit.edu,2,5.0,No,1,None
...,...,...,...,...,...,...,...,...
339,1,Siyoung Kim,ksiyoung@mit.edu,1,1.0,No,0,lny
340,2,Alina Yang,alinay@mit.edu,1,1.0,No,0,swotw
341,,Yina Wang,yina1123@mit.edu,1,1.0,No,0,None
342,2,Hasan Zeki Yildiz,hzyildiz@mit.edu,1,1.0,No,0,swotw
